Importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv("/Users/abhimanyuchettiar/Downloads/netflix_titles.csv")

Data cleaning

In [ ]:
# Handling missing values
df['director'] = df['director'].fillna('No Director')
df['cast'] = df['cast'].fillna('No Cast')
df['country'] = df['country'].fillna(df['country'].mode()[0])

In [ ]:
# Dropping duplicates and handling missing 'date_added' or 'rating'
df.dropna(subset=['date_added', 'rating'], inplace=True)

In [ ]:
# Formatting 'date_added' to datetime and extracting year/month
# 1. Convert to datetime, errors='coerce' turns problematic rows into NaT (Not a Time)
# We use .astype(str) first to ensure no float/NaN values break the string accessor
df['date_added'] = pd.to_datetime(df['date_added'].astype(str).str.strip(), errors='coerce')

# 2. Drop rows where date_added couldn't be parsed (the NaT rows)
df = df.dropna(subset=['date_added'])

# 3. Now extract your features safely
df['year_added'] = df['date_added'].dt.year.astype(int)
df['month_added'] = df['date_added'].dt.month_name()

In [ ]:
# cleaning duration to numeric for movies
# Check if x is a string before splitting to avoid the Float error
df['duration_num'] = df['duration'].apply(
    lambda x: int(x.split(' ')[0]) if isinstance(x, str) and 'min' in x else None
)

Data Analysis

Content Distribution (Movies vs TV Shows)

In [ ]:
type_counts = df['type'].value_counts().reset_index()
fig1 = px.pie(type_counts, values='count', names='type', 
             title='Distribution of Netflix Content',
             color_discrete_sequence=px.colors.sequential.RdBu)
fig1.show()

Growth of Content over time

In [ ]:
growth = df.groupby(['year_added', 'type']).size().reset_index(name='count')
fig2 = px.line(growth, x='year_added', y='count', color='type',
              title='Content Addition Trend Over Years',
              markers=True, template='plotly_dark')
fig2.show()

Top 10 Countries producing content

In [ ]:
# Splitting countries as some entries have multiple
top_countries = df['country'].str.split(', ').explode().value_counts().head(10).reset_index()
fig3 = px.bar(top_countries, x='count', y='country', orientation='h',
             title='Top 10 Content Producing Countries',
             color='count', color_continuous_scale='Reds')
fig3.show()

Contents Ratings Distribution

In [ ]:
ratings = df.groupby(['rating', 'type']).size().reset_index(name='count')
fig4 = px.sunburst(ratings, path=['type', 'rating'], values='count',
                  title='Content Rating Distribution by Type',
                  color='count', color_continuous_scale='Viridis')
fig4.show()

Movie Duration Analysis

In [ ]:
movies_df = df[df['type'] == 'Movie']
fig5 = px.histogram(movies_df, x='duration_num', nbins=50,
                   title='Distribution of Movie Durations',
                   labels={'duration_num': 'Duration (Minutes)'},
                   color_discrete_sequence=['indianred'])
fig5.show()

Top genres, world cloud style map

In [ ]:
# 1. Explode genres and count them
genres_df = df['listed_in'].str.split(', ').explode().value_counts().reset_index()

# 2. Rename columns explicitly to ensure 'Genre' and 'Count' exist
genres_df.columns = ['Genre', 'Count']

# 3. Take the top 15 for a cleaner visualization
top_genres = genres_df.head(15)

# 4. Create the Treemap using the new names
fig6 = px.treemap(top_genres, 
                 path=['Genre'], 
                 values='Count',
                 title='Top 15 Genres on Netflix',
                 color='Count', 
                 color_continuous_scale='Greens')

fig6.show()

implementing machine learning models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
ml_df = df[['type', 'rating', 'release_year', 'listed_in']].copy()

le_rating = LabelEncoder()
le_genre = LabelEncoder()
le_type = LabelEncoder()

ml_df['rating_encoded'] = le_rating.fit_transform(ml_df['rating'].astype(str))
ml_df['genre_encoded'] = le_genre.fit_transform(ml_df['listed_in'].astype(str))

X = ml_df[['release_year', 'rating_encoded', 'genre_encoded']]
y = le_type.fit_transform(ml_df['type'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_acc = accuracy_score(y_test, log_model.predict(X_test))

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt_model.predict(X_test))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_test))

# --- VISUALIZATION 1: Model Accuracy Comparison ---
models = ['Logistic Regression', 'Decision Tree', 'Random Forest']
accuracies = [log_acc, dt_acc, rf_acc]

fig_acc = px.bar(x=models, y=accuracies, color=accuracies,
                 title='Machine Learning Model Accuracy Comparison',
                 labels={'x': 'Model', 'y': 'Accuracy Score'},
                 color_continuous_scale='Viridis', range_y=[0, 1])
fig_acc.show()

# --- VISUALIZATION 2: Feature Importance (Random Forest) ---
importances = rf_model.feature_importances_
features = X.columns

fig_imp = px.bar(x=importances, y=features, orientation='h',
                 title='Feature Importance: What drives the Movie/TV Show classification?',
                 labels={'x': 'Importance Score', 'y': 'Feature'},
                 color=importances, color_continuous_scale='Reds')
fig_imp.show()

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)

print(f"SVM Accuracy: {svm_acc:.2f}")
cm = confusion_matrix(y_test, svm_preds)
class_names = le_type.inverse_transform([0, 1])

fig_cm = px.imshow(cm, 
                   text_auto=True, 
                   aspect="auto", 
                   labels=dict(x="Predicted", y="Actual", color="Count"),
                   x=class_names, 
                   y=class_names,
                   title='Confusion Matrix: SVM Performance',
                   color_continuous_scale='Purples')

fig_cm.update_layout(xaxis_title="Predicted Category", yaxis_title="Actual Category")
fig_cm.show()